In [20]:
from dotenv import load_dotenv
import wandb
import pickle
import os
import pandas as pd
from sklearn.model_selection import train_test_split

load_dotenv()

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "gp5")
WANDB_ENTITY = os.getenv("WANDB_ENTITY")

In [8]:
wandb.login(key=WANDB_API_KEY)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [9]:
datasets = dict()
for dir in ['dataset/train/images', 'dataset/train/bboxes', 'dataset/test/images', 'dataset/test/bboxes', 'dataset/val/images', 'dataset/val/bboxes']:
     datasets[dir] = os.listdir(dir)

In [10]:
datasets

{'dataset/train/images': ['347_0_chilli_wb_37.jpg',
  '428_5_grapes_wb_35.jpg',
  '372_5_tomato_wob_22.jpg',
  '167_0_grapes_wb_6.jpg',
  '298_3_grapes_wob_14.jpg',
  '418_3_apple_wb_46.jpg',
  '262_6_lemon_wob_5.jpg',
  '336_6_tomato_wob_35.jpg',
  '521_4_banana_wob_26.jpg',
  '363_2_tomato_wob_23.jpg',
  '397_2_grapes_wob_37.jpg',
  '87_5_apple_wb_1.jpg',
  '620_3_lemon_wob_13.jpg',
  '614_0_banana_wb_5.jpg',
  '609_2_chilli_wob_12.jpg',
  '470_3_raspberry - 18.jpg',
  '345_3_banana_wb_43.jpg',
  '141_6_banana_wob_15.jpg',
  '36_1_grapes_wb_3.jpg',
  '268_2_chilli_wb_4.jpg',
  '99_4_raspberry - 17.jpg',
  '441_0_lemon_wb_40.jpg',
  '344_4_lemon_wob_48.jpg',
  '635_2_tomato_wb_42.jpg',
  '300_2_grapes_wob_28.jpg',
  '621_4_chilli_wb_50.jpg',
  '449_3_banana_wob_42.jpg',
  '410_4_grapes_wb_23.jpg',
  '574_3_raspberry - 22.jpg',
  '504_2_apple_wb_23.jpg',
  '456_5_banana_wob_9.jpg',
  '20_5_lemon_wb_12.jpg',
  '289_4_tomato_wob_10.jpg',
  '450_0_chilli_wb_32.jpg',
  '50_2_lemon_wob_20.j

In [14]:
config = {
    "task": "dataset",
    "link": "https://www.kaggle.com/datasets/kvnpatel/fruits-vegetable-detection-for-yolov4"
}
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="dataset_vegetables", config=config)
pickle.dump(datasets, open(f"models/dataset.pkl", 'wb'))
artifact = wandb.Artifact(name="dataset", type="model", description=f"Пути к файлам в папках датасета")
artifact.add_file(f"models/dataset.pkl")
run.log_artifact(artifact)
run.finish()
wandb.finish()

wandb: WARNING Artifact "dataset" already exists with the same content. No new version will be created.


In [21]:
df = pd.read_csv('dataset/train.csv', sep="|")
df

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition,fraud
0,5,1054,54.70,7,0,3,0.027514,0.051898,0.241379,0
1,3,108,27.36,5,2,4,0.129630,0.253333,0.357143,0
2,3,1516,62.16,3,10,5,0.008575,0.041003,0.230769,0
3,6,1791,92.31,8,4,4,0.016192,0.051541,0.275862,0
4,5,430,81.53,3,7,2,0.062791,0.189605,0.111111,0
...,...,...,...,...,...,...,...,...,...,...
1874,1,321,76.03,8,7,2,0.071651,0.236854,0.347826,0
1875,1,397,41.89,5,5,0,0.065491,0.105516,0.192308,1
1876,4,316,41.83,5,8,1,0.094937,0.132373,0.166667,0
1877,2,685,62.68,1,6,2,0.035036,0.091504,0.041667,0


In [18]:
y = df["fraud"]
X = df.drop(columns=["fraud"])

In [22]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [23]:
X_train["fraud"] = y_train
X_val["fraud"] = y_val

X_train.to_csv('dataset/train_new.csv')
X_val.to_csv('dataset/val_new.csv')

In [24]:
X_test = pd.read_csv('dataset/test.csv')
y_test = pd.read_csv("dataset/DMC-2019-realclass.csv", sep="|")["fraud"]

X_test['fraud'] = y_test
X_test.to_csv('dataset/test_new.csv')

In [25]:
config = {
    "task": "dataset",
    "link": "https://www.kaggle.com/code/eastonlarson/fraud-detection-exploratory-analysis"
}
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="dataset_vegetables", config=config)
artifact = wandb.Artifact(name="dataset_fraud", type="model", description=f"Объекты в тестовой, валидационной и обучающей выборке")
artifact.add_file(f"dataset/train_new.csv")
artifact.add_file(f"dataset/val_new.csv")
artifact.add_file("dataset/test_new.csv")
run.log_artifact(artifact)
run.finish()
wandb.finish()